In [ ]:
%pip install prophet

In [ ]:
import pandas as pd
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

In [ ]:
%pip install scikit-learn

In [ ]:
# Load prepped data

df = pd.read_csv("foods_3_305_daily.csv", parse_dates=["date"])

# Prophet requires columns named exactly 'ds' (date) and 'y' (target)
prophet_df = df.rename(columns={"date": "ds", "units_sold": "y"})


In [ ]:

prophet_df["price_baseline"] = prophet_df["avg_price"].rolling(90, min_periods=1).mean()
prophet_df["price_ratio"] = prophet_df["avg_price"] / prophet_df["price_baseline"]
# price_ratio ~1.0 = normal price, <1.0 = discounted, >1.0 = above-normal


In [ ]:
# Train/test split. last 28 days as test by standard
TEST_DAYS = 28
train = prophet_df.iloc[:-TEST_DAYS].copy()
test = prophet_df.iloc[-TEST_DAYS:].copy()

In [ ]:
#  Build model with regressors, Prophet handles weekly/yearly seasonality automatically, We add price_ratio (promo depth) and event/SNAP flags as regressors.

model = Prophet(
    growth="flat",  # no secular trend expected for a single SKU — avoids
                     # Prophet extrapolating a spurious late-training dip/spike
    weekly_seasonality=True,
    yearly_seasonality=True,
    daily_seasonality=False,
)
model.add_regressor("price_ratio")
model.add_regressor("has_event")
model.add_regressor("any_snap")
 
train_reg = train[["ds", "y", "price_ratio", "has_event", "any_snap"]]
model.fit(train_reg)


In [ ]:
# 4. Forecast the test period
#    Prophet needs the regressor VALUES for future dates too —
#    we already have them since this is historical test data.
# ---------------------------------------------------------
future = test[["ds", "price_ratio", "has_event", "any_snap"]]
forecast = model.predict(future)


In [ ]:
# DIAGNOSTIC: figure out what's driving negative predictions
# before we clip them away. Print the component breakdown for
# the first few test rows.
# ---------------------------------------------------------
diag_cols = ["ds", "trend", "weekly", "yearly", "extra_regressors_additive", "yhat"]
diag_cols = [c for c in diag_cols if c in forecast.columns]
print("\n--- DIAGNOSTIC: forecast component breakdown (pre-clip) ---")
print(forecast[diag_cols].head(10).to_string(index=False))
print("--- end diagnostic ---\n")

# Demand can't be negative — clip any residual negative predictions
forecast["yhat"] = forecast["yhat"].clip(lower=0)


In [ ]:
# 5. Evaluate
# ---------------------------------------------------------
y_true = test["y"].values
y_pred = forecast["yhat"].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Prophet Test MAE:  {mae:.2f}")
print(f"Prophet Test RMSE: {rmse:.2f}")

# Save forecast vs actual for later comparison with XGBoost
results = test[["ds", "y"]].copy()
results["prophet_pred"] = y_pred
results.to_csv("prophet_results.csv", index=False)
print("\nSaved prophet_results.csv for later comparison")
print(results.head(10))


In [ ]:
# 6. Plot (optional but useful for your writeup/dashboard)
# ---------------------------------------------------------
fig1 = model.plot(forecast)
fig1.savefig("prophet_forecast_plot.png")

fig2 = model.plot_components(forecast)
fig2.savefig("prophet_components_plot.png")
print("\nSaved prophet_forecast_plot.png and prophet_components_plot.png")
